# 07 — Initial Bias Analysis

Quick initial look at flag-rate disparities across name origins using `full_results2.db` (SQLite — LT run with `language=auto`, language-all JAR).

**Three primary comparison conditions (Condition B — Latin transliteration):**
- `hunspell_latin_known`
- `symspell_latin_known`
- `lt_auto_latin_known`

**LT-auto additionally compared on Condition A (original script):**
- `lt_auto_orig_known` — uses per-token language detection across ~50 dictionaries, so unlike the other tools' Orig conditions it is meaningful for both Latin and non-Latin scripts where the language is supported.

**Key considerations:**
1. **Cross-tool apples-to-apples:** Condition B (Latin transliteration) puts all three tools on the same input. LT-auto Orig is included as a separate track rather than a direct peer.
2. **Multi-word / noise names:** names with spaces, digits, or punctuation produce artefactual results and are filtered before analysis.
3. **Scope of 'known':** a name is flagged because it is not in the tool's dictionary — not because of any name-specific knowledge. Bias is measured as *disparity in flagging rates across origin groups*.

All visualisations show both the **full dataset** and the **cleaned single-word dataset** where the comparison justifies the cleaning choice, then **cleaned only** thereafter.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.1f}'.format)

COLS = [
    'name', 'name_script', 'top_country', 'top_country_prob',
    'ethnicolr_race',
    'hunspell_orig_known',  'hunspell_latin_known',
    'symspell_orig_known',  'symspell_latin_known',
    'lt_auto_orig_known',   'lt_auto_latin_known',
]

from sqlalchemy import create_engine

engine = create_engine('sqlite:///../data/full_results2.db')
df = pd.read_sql(
    f"""SELECT {', '.join(COLS)} FROM names""",
    engine,
)

BOOL_COLS = [c for c in COLS if c.endswith('_known')]
df[BOOL_COLS] = df[BOOL_COLS].astype(bool)

# Condition B cross-tool comparison (apples-to-apples: all on Latin transliteration)
PRIMARY_B = [
    ('Hunspell (Latin)', 'hunspell_latin_known'),
    ('SymSpell (Latin)', 'symspell_latin_known'),
    ('LT-auto (Latin)',  'lt_auto_latin_known'),
]

# Extended: adds LT-auto Orig as a 4th condition alongside the Condition B comparisons.
# LT-auto uses language=auto (language-all JAR), so Orig attempts per-token language
# detection across ~50 dictionaries — meaningful for both Latin and non-Latin scripts
# where the language is supported, not just a silent pass-through.
EXTENDED = PRIMARY_B + [('LT-auto (Orig)', 'lt_auto_orig_known')]

print(f'Loaded {len(df):,} names')

Loaded 727,352 names


## 1. Cleaning

The cleaned dataset keeps single-word names including:
- **accented / diacritic letters** (é, ñ, ü, ø — legitimate name characters)
- **non-Latin scripts** (Arabic, CJK, Cyrillic, etc. — pass through unchanged)
- **hyphens and apostrophes** (O'Brien, Mary-Jane)

Removed: whitespace (multi-word names), digits, and noise punctuation (`. * _ @ # $ % ^` etc.)

In [ ]:
# Hyphens and apostrophes are intentionally excluded from noise
# as they appear in legitimate names (O'Brien, Mary-Jane).
NOISE = r"[\s*._0-9@#$%^&+=|<>{}\[\]\\;:,!?~/'\"]"

mask_clean = ~df['name'].str.contains(NOISE, regex=True, na=True)
df_clean = df[mask_clean].copy()

removed = len(df) - len(df_clean)
print(f'Full dataset:    {len(df):,}')
print(f'Cleaned dataset: {len(df_clean):,}  ({removed:,} removed, {100*removed/len(df):.1f}%)')
print()
print('Script distribution (cleaned):')
print(df_clean['name_script'].value_counts().to_string())

## 2. LT-auto behaviour across script families

This dataset uses `language=auto` (LT language-all JAR), which attempts per-token language detection before applying spell-check rules. LT supports ~50 languages including several with non-Latin scripts (Arabic, Chinese, Russian/Cyrillic, Greek, Japanese, etc.).

Unlike the `en-US` run (where non-Latin characters were silently passed as 'known' because Morfologik only applied to Latin script), LT-auto may genuinely engage with non-Latin script names where the script's language is in its dictionary set. For unsupported scripts it will still default to no match.

The table below shows the flagging rates empirically across script families for all conditions. The LT-auto Orig column for non-Latin scripts is now a real observation — not a known artefact to discount.

In [ ]:
ALL_CONDITIONS = [
    ('Hunspell (Orig)',  'hunspell_orig_known'),
    ('Hunspell (Latin)', 'hunspell_latin_known'),
    ('SymSpell (Orig)',  'symspell_orig_known'),
    ('SymSpell (Latin)', 'symspell_latin_known'),
    ('LT-auto (Orig)',   'lt_auto_orig_known'),
    ('LT-auto (Latin)',  'lt_auto_latin_known'),
]

def scope_table(frame):
    lat = frame['name_script'] == 'Latin'
    rows = []
    for cond, col in ALL_CONDITIONS:
        for grp_label, mask in [('Latin scripts', lat), ('Non-Latin scripts', ~lat)]:
            sub = frame[mask]
            flagged = (~sub[col]).sum()
            rows.append({'Condition': cond, 'Scripts': grp_label,
                         '% flagged': round(100 * flagged / len(sub), 1)})
    tbl = pd.DataFrame(rows).pivot(
        index='Condition', columns='Scripts', values='% flagged')
    tbl.columns.name = None
    return tbl

print('=== FULL DATASET ===')
display(scope_table(df))
print()
print('=== CLEANED DATASET ===')
display(scope_table(df_clean))

## 3. Overall flag rates — Condition B (Latin)

Headline flag rates for the three primary conditions across full and cleaned datasets.

In [ ]:
def headline_table(frame, label):
    n = len(frame)
    rows = []
    for cond, col in EXTENDED:
        known   = frame[col].sum()
        flagged = n - known
        rows.append({'Condition': cond, 'n total': f'{n:,}',
                     'n known': f'{known:,}', 'n flagged': f'{flagged:,}',
                     '% flagged': f'{100 * flagged / n:.1f}%'})
    tbl = pd.DataFrame(rows).set_index('Condition')
    tbl.index.name = label
    return tbl

print('=== FULL DATASET ===')
display(headline_table(df, 'Full'))
print()
print('=== CLEANED DATASET ===')
display(headline_table(df_clean, 'Cleaned'))

## 4. Flag rate by script family — Condition B

For Condition B all names are in the Latin alphabet, so the script label here reflects the name's *original* writing system — a proxy for how different the transliterated form looks from English words.

In [ ]:
def script_breakdown(frame, min_n=100):
    rows = []
    for script, grp in frame.groupby('name_script', observed=True):
        if len(grp) < min_n:
            continue
        row = {'Script': script, 'n': len(grp)}
        for lbl, col in EXTENDED:
            row[f'{lbl} flagged%'] = round(100 * (1 - grp[col].mean()), 1)
        rows.append(row)
    return (pd.DataFrame(rows)
            .sort_values('Hunspell (Latin) flagged%', ascending=False)
            .set_index('Script'))

print('=== FULL DATASET ===')
display(script_breakdown(df))
print()
print('=== CLEANED DATASET ===')
display(script_breakdown(df_clean))

## 5. Flag rate by ethnicolr category — Condition B (all scripts)

ethnicolr predicts name origin from a 13-category taxonomy. The `ethnicolr_race` column uses hierarchical labels (e.g. `GreaterEuropean,British`); the leaf category is used here.

Includes all script families — non-Latin names are assessed on their transliterated form for Condition B.

---
Sections 3 and 4 confirm that removing multi-word and noise names does not materially shift the script distribution or headline flag rates — the patterns are consistent between full and cleaned datasets. All analysis from this point uses the **cleaned dataset only**.

In [ ]:
def eth_breakdown(frame, min_n=500):
    f = frame.copy()
    f['eth_cat'] = f['ethnicolr_race'].str.split(',').str[-1].str.strip()
    rows = []
    for cat, grp in f.groupby('eth_cat', observed=True):
        if len(grp) < min_n:
            continue
        row = {'Category': cat, 'n': len(grp)}
        for lbl, col in EXTENDED:
            row[f'{lbl} flagged%'] = round(100 * (1 - grp[col].mean()), 1)
            row[f'{lbl} known%']   = round(100 * grp[col].mean(), 1)
        rows.append(row)
    return (pd.DataFrame(rows)
            .sort_values('Hunspell (Latin) flagged%', ascending=False)
            .reset_index(drop=True))

eth_clean = eth_breakdown(df_clean)
display(eth_clean.set_index('Category'))

In [ ]:
def eth_bar(ax, data, col_suffix='flagged%'):
    cats = data['Category'].tolist()
    x = np.arange(len(cats))
    n_conds = len(EXTENDED)
    w = 0.8 / n_conds
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
    for i, (lbl, _) in enumerate(EXTENDED):
        # LT-auto (Orig) hatched to distinguish it from the Condition B bars
        hatch = '//' if lbl == 'LT-auto (Orig)' else None
        ax.bar(x + i * w, data[f'{lbl} {col_suffix}'], width=w,
               label=lbl, color=colors[i % len(colors)], hatch=hatch)
    ax.set_xticks(x + w * (n_conds - 1) / 2)
    ax.set_xticklabels(cats, rotation=40, ha='right', fontsize=8)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.legend(fontsize=7)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 5), sharey=False)
eth_bar(ax1, eth_clean, col_suffix='flagged%')
eth_bar(ax2, eth_clean, col_suffix='known%')
ax1.set_ylabel('% flagged')
ax1.set_title('% flagged by origin category')
ax2.set_ylabel('% recognised (not flagged)')
ax2.set_title('% recognised by origin category')
fig.suptitle(
    'Condition B (Latin) cross-tool comparison + LT-auto Orig — ethnicolr categories  |  Cleaned dataset',
    fontsize=10
)
plt.tight_layout()
plt.show()

## 6. Isolating lexical/phonological bias — Latin-script names only

Restricting to names that are **already in Latin script** removes transliteration artefacts entirely. For these names `name == name_latin`, so Condition A and B are identical.

Any remaining flag-rate disparity across origin groups is purely **lexical/phonological bias** — the names look different from English words even though they use the same alphabet. This is the cleanest test of systemic bias.

In [ ]:
def lat_breakdown(frame, min_n=200):
    f = frame[frame['name_script'] == 'Latin'].copy()
    f['eth_cat'] = f['ethnicolr_race'].str.split(',').str[-1].str.strip()
    rows = []
    for cat, grp in f.groupby('eth_cat', observed=True):
        if len(grp) < min_n:
            continue
        row = {'Category': cat, 'n': len(grp)}
        for lbl, col in EXTENDED:
            row[f'{lbl} flagged%'] = round(100 * (1 - grp[col].mean()), 1)
            row[f'{lbl} known%']   = round(100 * grp[col].mean(), 1)
        rows.append(row)
    return (pd.DataFrame(rows)
            .sort_values('Hunspell (Latin) flagged%', ascending=False)
            .reset_index(drop=True))

lat_clean = lat_breakdown(df_clean)
display(lat_clean.set_index('Category'))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 5), sharey=False)
eth_bar(ax1, lat_clean, col_suffix='flagged%')
eth_bar(ax2, lat_clean, col_suffix='known%')
ax1.set_ylabel('% flagged')
ax1.set_title('% flagged by origin category')
ax2.set_ylabel('% recognised (not flagged)')
ax2.set_title('% recognised by origin category')
fig.suptitle(
    'Condition B (Latin) cross-tool + LT-auto Orig — Latin-script names only  |  Cleaned dataset\n'
    'Differences reflect lexical/phonological bias. LT-auto Orig here shows language-detection benefit only.',
    fontsize=10
)
plt.tight_layout()
plt.show()

## 7. Tool agreement — Condition B

Consistent flagging by all three tools is stronger evidence of systemic bias than any single tool flagging a name.

## 7. LT-auto: language-detection effect — Orig vs Latin delta

LT-auto uses per-token language detection across ~50 dictionaries. This means `lt_auto_orig_known` may differ from `lt_auto_latin_known` for two distinct reasons:

- **Non-Latin scripts:** LT-auto may genuinely engage with these names if the script's language is in its dictionary (Arabic, Chinese, Russian, Greek, Japanese etc.). The Orig vs Latin delta here shows whether LT-auto's language detection adds recognition beyond what anyascii transliteration produces.
- **Latin-script names with diacritics:** LT-auto may recognise "José" through its Spanish dictionary where anyascii produces "Jose". The delta here shows how much diacritic-preservation matters for LT's recognition.

The first table below shows the Orig − Latin delta per script across all scripts (full dataset) — this shows what LT-auto actually does when it encounters each script. The second focuses on Latin-script cleaned names to isolate the diacritic/language-detection effect without script change.

In [ ]:
def lt_delta_script(frame, min_n=100):
    rows = []
    for script, grp in frame.groupby('name_script', observed=True):
        if len(grp) < min_n:
            continue
        orig_known  = round(100 * grp['lt_auto_orig_known'].mean(), 1)
        latin_known = round(100 * grp['lt_auto_latin_known'].mean(), 1)
        rows.append({
            'Script': script,
            'n': len(grp),
            'LT Orig known%':  orig_known,
            'LT Latin known%': latin_known,
            'Orig - Latin (pp)': round(orig_known - latin_known, 1),
        })
    return (pd.DataFrame(rows)
            .sort_values('Orig - Latin (pp)', ascending=False)
            .set_index('Script'))

print('All scripts (full dataset) — shows LT-auto actual engagement with each script family:')
display(lt_delta_script(df))
print()
print('Latin-script only (cleaned) — isolates diacritic/language-detection effect:')
display(lt_delta_script(df_clean[df_clean['name_script'] == 'Latin']))

In [ ]:
# Per-origin-category delta for Latin-script names only
def lt_delta_eth(frame, min_n=200):
    f = frame[frame['name_script'] == 'Latin'].copy()
    f['eth_cat'] = f['ethnicolr_race'].str.split(',').str[-1].str.strip()
    rows = []
    for cat, grp in f.groupby('eth_cat', observed=True):
        if len(grp) < min_n:
            continue
        orig_known  = round(100 * grp['lt_auto_orig_known'].mean(), 1)
        latin_known = round(100 * grp['lt_auto_latin_known'].mean(), 1)
        rows.append({
            'Category': cat,
            'n': len(grp),
            'LT Orig known%':  orig_known,
            'LT Latin known%': latin_known,
            'Orig - Latin (pp)': round(orig_known - latin_known, 1),
        })
    return (pd.DataFrame(rows)
            .sort_values('Orig - Latin (pp)', ascending=False)
            .set_index('Category'))

display(lt_delta_eth(df_clean))

# Categories with a large positive delta have names that LT recognises via their
# native-language dictionary but which lose recognition after anyascii transliteration.
# Categories with near-zero delta have names where language detection adds little.

In [ ]:
def agreement_table(frame):
    h  = frame['hunspell_latin_known']
    s  = frame['symspell_latin_known']
    lt = frame['lt_auto_latin_known']
    n  = len(frame)
    n_known = h.astype(int) + s.astype(int) + lt.astype(int)
    patterns = {
        'All 3 known':           ( h &  s &  lt).sum(),
        'All 3 flagged':         (~h & ~s & ~lt).sum(),
        'Hunspell only flagged': (~h &  s &  lt).sum(),
        'SymSpell only flagged': ( h & ~s &  lt).sum(),
        'LT-auto only flagged':  ( h &  s & ~lt).sum(),
        'Two tools flagged':     (n_known == 1).sum(),
    }
    rows = [{'Pattern': k, 'n': f'{v:,}',
             '% of total': f'{100 * v / n:.1f}%'}
            for k, v in patterns.items()]
    return pd.DataFrame(rows).set_index('Pattern')

display(agreement_table(df_clean))

In [ ]:
# Among names flagged by ALL 3 tools, what fraction comes from each origin category?
# This is the most conservative proxy for systemic bias.
def all_flagged_breakdown(frame, min_n=500):
    f = frame.copy()
    f['eth_cat'] = f['ethnicolr_race'].str.split(',').str[-1].str.strip()
    all_flagged = (
        ~f['hunspell_latin_known']
        & ~f['symspell_latin_known']
        & ~f['lt_auto_latin_known']
    )
    rows = []
    for cat, grp in f.groupby('eth_cat', observed=True):
        n_cat = len(grp)
        if n_cat < min_n:
            continue
        n_af = all_flagged[grp.index].sum()
        rows.append({
            'Category': cat,
            'n in category': f'{n_cat:,}',
            '% flagged by all 3': round(100 * n_af / n_cat, 1),
        })
    return (pd.DataFrame(rows)
            .sort_values('% flagged by all 3', ascending=False)
            .set_index('Category'))

display(all_flagged_breakdown(df_clean))